# **Prompting LLMs**

**Author:** Louis G. Binwag III

**Reference:** Course Notes (.ipynb)

---

**Instructions:** Evaluate the models on a subset of the BelebeleLinks to an external site. benchmark, specifically the Filipino subset. This task will require you to work with Filipino prompts.


1. **Load the dataset:** The Belebele dataset is available on HuggingFaceLinks to an external site.. You will need to load the tgl_Latn split (all 900 rows).

2. **Create Filipino prompts:** For each example in the dataset, you will need to construct a multiple-choice question in Filipino. The dataset contains a context, a question, and four possible answers.

3. **Evaluate the models:** Use the gemma3:1b, llama3.2:1b, and the quantized aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k models to answer the questions.

4. **Save the results:** Your output should be one JSONL results file for each model you test (e.g., belebele_results_gemma3:1b.jsonl). Each line in the file should be a JSON object containing the model_name, prompt, response, correct_answer, and whether the model's answer was correct.

**Expected Output:** Jupyter Notebook

## **Imports**

In [ ]:
!pip install --quiet datasets

In [ ]:
# !wsl --install
# !curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!ollama pull gemma3:1b
!ollama pull llama3.2:1b
!ollama pull aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k


In [ ]:
!ollama list

In [ ]:
import subprocess
import re
import os
import asyncio
import pandas as pd

import json
import re
import subprocess
from tqdm import tqdm

## **Loading Dataset**

In [ ]:
from datasets import load_dataset

ds = load_dataset("facebook/belebele", "tgl_Latn")
split = 'validation' if 'validation' in ds else 'test' if 'test' in ds else 'train'

In [ ]:
df = ds[split].to_pandas()
df.head()

In [52]:
def build_prompt(item):
    """Builds a Filipino multiple-choice question prompt."""
    passage = item["flores_passage"]
    question = item["question"]
    a1 = item["mc_answer1"]
    a2 = item["mc_answer2"]
    a3 = item["mc_answer3"]
    a4 = item["mc_answer4"]

    prompt = (
        "Basahin ang sipi at sagutin ang tanong.\n\n"
        f"Flores Passage: {passage}\n\n"
        f"Question: {question}\n\n"
        "Choices:\n"
        f"(A) {a1}\n"
        f"(B) {a2}\n"
        f"(C) {a3}\n"
        f"(D) {a4}\n\n"
        "Ang tamang sagot ay:"
    )

    return prompt


### **Starting Ollama Server**

In [57]:
# Set LD_LIBRARY_PATH so the system can find ollama's shared libraries
os.environ['LD_LIBRARY_PATH'] = '/usr/lib/x86_64-linux-gnu'

async def run_ollama():
    proc = await asyncio.create_subprocess_shell(
        'ollama serve',
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )
    # The server is now running in the background
    print("Ollama server started.")
    # We don't await proc.communicate() here to let it run in the background

# Start the server
await run_ollama()

# Give the server a moment to start
!sleep 5

Ollama server started.


In [ ]:
def ask_ollama(prompt, model="gemma3:1b"):
    """
    Sends the prompt to the Ollama model and returns raw text output.
    """
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    return result.stdout.decode("utf-8").strip()

In [ ]:
sample_item = df.iloc[0]
prompt = build_prompt(sample_item)

print(ask_ollama(prompt, "gemma3:1b"))
print(ask_ollama(prompt, "llama3.2:1b"))
print(ask_ollama(prompt, "aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k"))



## Evaluation Code

In [ ]:
def ask_ollama(prompt, model):
    """
    Sends the prompt to the specified Ollama model and returns raw text output.
    """
    try:
        result = subprocess.run(
            ["ollama", "run", model],
            input=prompt.encode("utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=120
        )
        return result.stdout.decode("utf-8").strip()
    except subprocess.TimeoutExpired:
        return "[ERROR: Timeout]"
    except Exception as e:
        return f"[ERROR: {e}]"



def clean_response(text):
    """
    Extracts only the first (A/B/C/D) from the model's response.
    """
    match = re.search(r"[ABCD]", text.upper())
    return f"({match.group(0)})" if match else None



def map_correct_answer(num):
    """
    Converts correct_answer_num (1–4) to corresponding letter.
    """
    mapping = {1: "(A)", 2: "(B)", 3: "(C)", 4: "(D)"}
    return mapping.get(num, None)



def evaluate_model(df, model_name):
    """
    Evaluates and outputs:
    
    model_name, prompt, response, correct_answer (1–4), is_correct
    """
    results = []

    for _, item in tqdm(df.iterrows(), total=len(df)):
        prompt = build_prompt(item)
        response = ask_ollama(prompt, model=model_name)
        parsed = clean_response(response)

        correct_num = (item.get("correct_answer_num"))

        if pd.notnull(correct_num):
            correct_num = int(correct_num)
        else:
            correct_num = None

        correct_letter = map_correct_answer(correct_num)
        is_correct = parsed == correct_letter if correct_letter else False

        results.append({
            "model_name": model_name,
            "prompt": prompt,
            "response": clean_response(response),
            "correct_answer": map_correct_answer(correct_num), 
            "is_correct": is_correct
        })

    """
    Save to JSONL
    """

    out_file = f"belebele_results_{model_name.replace('/', '_')}.jsonl"
    with open(out_file, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"Saved {out_file}")
    return results



## Evaluation

In [ ]:
test_df = df.head(2)   

# print(test_df)

print(evaluate_model(test_df, "gemma3:1b"))
print(evaluate_model(test_df, "llama3.2:1b"))
print(evaluate_model(test_df, "aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k"))


In [ ]:
results_gemma = evaluate_model(df, "gemma3:1b")

In [ ]:
results_llama = evaluate_model(df, "llama3.2:1b")

In [ ]:
results_sealion = evaluate_model(df, "aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k")

*Results saved as .jsonl file (belebele_results_[model].jsonl)*